# noise

In [ ]:
import galsim
import matplotlib.pyplot as pyplot
import numpy
from functions_images import build_full_image, build_psf, get_constants, get_offset, unpack_lhs
from functions_lhs import adding_cosmos_catalog_galaxies, draw_lhs_gal, draw_lhs_img
from functions_projections import project_lhs
from tqdm import tqdm
from utils import DIRS

GSPARAMS = galsim.GSParams(
    xvalue_accuracy=2.0e-4, kvalue_accuracy=2.0e-4, maxk_threshold=5.0e-3, folding_threshold=1.0e-2
)

In [ ]:
NB_OF_SAMPLES = 3
ELLIPS_STYLE = "gaussian"
PIXELS_OUT = False
LHS_POS = False
SEED_DATABASE = 2222
PERCENTAGE_COSMOS = 20
TYPE_COSMOS = "real"  # "real" or "parametric"

PIXEL_SCALE, PATCH_SIZE, IMG_SIZE, EXPTIME, GAIN, readoutnoise, SKY_BKG, ZP = get_constants()

In [ ]:
# create lhs image
lhs_img = draw_lhs_img(seed=SEED_DATABASE * 1_000, n_samples=NB_OF_SAMPLES)

# create lhs galaxies
lhs_gal = draw_lhs_gal(LHS_POS, SEED_DATABASE, NB_OF_SAMPLES * 10)

# lhs galaxies projection
lhs_gal = project_lhs(hypercube=lhs_gal, ellips_style=ELLIPS_STYLE, pixel_out=PIXELS_OUT)

# taking real/parametric galaxies from cosmos catalogs
nb_galaxies_cosmos = int(PERCENTAGE_COSMOS * NB_OF_SAMPLES * 10 / 100)

numpy.random.seed(22)
replace_indices = numpy.random.choice(a=NB_OF_SAMPLES * 10, size=nb_galaxies_cosmos, replace=False)

lhs_gal = adding_cosmos_catalog_galaxies(lhs_gal, numpy.sort(replace_indices), version="9999")

psf = build_psf("airy")

In [ ]:
stamps = []
for i in tqdm(range(lhs_img.shape[0]), desc="Rendering galaxy scenes"):
    # sigma_noise = []
    img_params = lhs_img[i]
    gal_params = lhs_gal[i * 10 : (i + 1) * 10]

    params = unpack_lhs([img_params, gal_params])
    avg_g1 = params["g1"]
    avg_g2 = params["g2"]

    full_image = build_full_image(IMG_SIZE, PIXEL_SCALE)
    noise_image = build_full_image(IMG_SIZE, PIXEL_SCALE)
    flux = EXPTIME / GAIN * 10 ** (-(params["mag"] - ZP) / 2.5)

    for i in range(10):
        # compute Sersic profile with cosmos
        if params["index_cosmos"][i] > 0:
            CHOICE = int(params["index_cosmos"][i])
            SAMPLE = str(params["sample_cosmos"][i])

            cosmos_cat = galsim.COSMOSCatalog(
                sample=SAMPLE,
                dir=DIRS["cosmos"],
                exclusion_level="none",
                exptime=EXPTIME,
            )

            # gal = cosmos_cat.makeGalaxy(
            #     gal_type=TYPE_COSMOS,
            #     index=CHOICE,
            #     noise_pad_size=5,
            #     sersic_prec=0.0,
            #     gsparams=GSPARAMS,
            # )

            # gal_image, *_ = cosmos_cat.getRealParams(CHOICE)
            # gal = galsim.InterpolatedImage(
            #     gal_image, scale=PIXEL_SCALE, normalization="flux", gsparams=GSPARAMS
            # )
            # BRIGHTNESS_BOOST = 1
            # gal = gal.withFlux(flux[i] * BRIGHTNESS_BOOST)

        else:
            # compute Sersic profile with galsim
            gal = galsim.Sersic(
                n=params["sersic"][i],
                half_light_radius=params["hlr"][i],
                flux=flux[i],
                gsparams=GSPARAMS,
                trunc=params["hlr"][i] * 4.5,
            )

            # apply ellips
            gal = gal.shear(e1=params["int_e1"][i], e2=params["int_e2"][i])

        # apply shear
        gal = gal.shear(g1=avg_g1, g2=avg_g2)  # simple case

        # convolve galaxy with PSF
        final = galsim.Convolve([psf, gal])

        # offset
        image_pos, ix_nominal, iy_nominal, offset = get_offset(
            params["off_x"][i], params["off_y"][i], params["x"][i], params["y"][i]
        )

        # draw galaxy
        stamp = galsim.ImageF(IMG_SIZE, IMG_SIZE, scale=PIXEL_SCALE)
        final.drawImage(image=stamp, wcs=full_image.wcs.local(image_pos), offset=offset)
        stamp.setCenter(ix_nominal, iy_nominal)

        # add stamp to the main image
        bounds = stamp.bounds & full_image.bounds
        full_image[bounds] += stamp[bounds]

    # gaussian noise image
    F_SKY = PIXEL_SCALE ** (2) * EXPTIME * 10 ** (-(SKY_BKG - ZP) / 2.5)  # e-/pixel
    variance = (numpy.sqrt(readoutnoise**2 + F_SKY) / GAIN) ** 2  # e- -> ADU
    noise = galsim.GaussianNoise(rng=galsim.BaseDeviate(22), sigma=numpy.sqrt(variance))
    noise_image.addNoise(noise)

    # # add noise
    full_image += noise_image
    full_image_array = full_image.array
    stamps.append(full_image_array)

In [ ]:
# name = "parametric_no_noise"
# name = "parametric_noise"
# name = "real_no_noise"
# name = "real_noise"
name = "real_noise_0"

In [ ]:
fig, axes = pyplot.subplots(1, 3, figsize=(18, 6))
for idx, ax in enumerate(axes.flat):
    im = ax.imshow(numpy.arcsinh(stamps[idx]), origin="lower")
    # im = ax.imshow((stamps[idx]), origin="lower")
    ax.set_title(f"Example {idx + 1}")
    fig.colorbar(im, ax=ax, shrink=0.75)
fig.tight_layout()
fig.savefig(DIRS["images_cosmos"] / f"{name}.pdf", dpi=300, bbox_inches="tight")
fig.show()

# gal image

In [ ]:
# _, _, galaxy_variance = cosmo_cat.real_cat.getNoiseProperties(CHOICE)
# sigma_noise = numpy.sqrt(galaxy_variance)

In [ ]:
CHOICE = 10

cosmo_cat = galsim.COSMOSCatalog(sample="23.5", dir=DIRS["cosmos"], exclusion_level="none")
gal_image, *_ = cosmo_cat.getRealParams(CHOICE)

pyplot.figure(figsize=(6, 6))
pyplot.imshow(gal_image.array, origin="lower")
pyplot.colorbar(label="Flux")
pyplot.tight_layout()
pyplot.show()

# test real para

In [ ]:
CHOICE = 10
SAMPLE = "23.5"
NOISE_PAD_SIZE = 0
TYPE_COSMOS = "real"  # "real" or "parametric"
PIXEL_SCALE, XSIZE, YSIZE, IMG_SIZE, EXPTIME, GAIN, readoutnoise, SKY_BKG, ZP = get_constants()

test_cosmos_cat = galsim.COSMOSCatalog(
    sample=SAMPLE,
    dir=DIRS["cosmos"],
    use_real=True,
    exclusion_level="none",
    exptime=EXPTIME,
)

common_kwargs = dict(index=CHOICE, noise_pad_size=0, deep=False, sersic_prec=0.0, gsparams=GSPARAMS)

real_gal = test_cosmos_cat.makeGalaxy(gal_type="real", **common_kwargs)
param_gal = test_cosmos_cat.makeGalaxy(gal_type="parametric", **common_kwargs)

psf = build_psf("airy")

real_gal = galsim.Convolve(real_gal, psf)
im_real = galsim.Image(XSIZE, YSIZE)
real_gal.drawImage(image=im_real, scale=PIXEL_SCALE)

param_gal = galsim.Convolve(param_gal, psf)
im_param = galsim.Image(XSIZE, YSIZE)
param_gal.drawImage(image=im_param, scale=PIXEL_SCALE)

fig, axs = pyplot.subplots(1, 2, figsize=(8, 4))

im0 = axs[0].imshow(im_real.array, origin="lower")
axs[0].set_title(f"Real Galaxy {CHOICE}")
fig.colorbar(im0, ax=axs[0], orientation="vertical", fraction=0.05, pad=0.05)
im1 = axs[1].imshow(im_param.array, origin="lower")
axs[1].set_title(f"Parametric Model {CHOICE}")
fig.colorbar(im1, ax=axs[1], orientation="vertical", fraction=0.05, pad=0.05)

fig.tight_layout()
fig.show()

# matched

In [ ]:
from functions_lhs import draw_lhs_gal, normalise, euclidian_style, maha_style
from functions_projections import project_lhs
import numpy
from plotly import subplots, graph_objects
from collections import defaultdict
from pathlib import Path
from pandas import read_csv
from utils import DIRS

In [ ]:
def check_dupicates(arr):
    row_groups = defaultdict(list)

    for idx, row in enumerate(arr):
        row_key = tuple(row)
        row_groups[row_key].append(idx)

    duplicates = {row: indices for row, indices in row_groups.items() if len(indices) > 1}
    return duplicates

In [ ]:
NB_OF_SAMPLES = 1
lhs_gal = draw_lhs_gal(False, 2222, NB_OF_SAMPLES * 10)
lhs_gal = project_lhs(hypercube=lhs_gal, ellips_style="gaussian", pixel_out=False)

PERCENTAGE_COSMOS = 50
nb_galaxies_cosmos = int(PERCENTAGE_COSMOS * NB_OF_SAMPLES * 10 / 100)
numpy.random.seed(22)
replace_indices = numpy.sort(
    numpy.random.choice(a=NB_OF_SAMPLES * 10, size=nb_galaxies_cosmos, replace=False)
)

version = "9999"

SCALING = "min_max"
NORMA = "euclid"

new_columns = numpy.zeros((lhs_gal.shape[0], 2), dtype=lhs_gal.dtype) - 1
lhs_gal = numpy.hstack((lhs_gal, new_columns))

check_columns = [0, 1, 4, 5, 6]  # e1 e2 mag hlr sersic
target = lhs_gal[:, check_columns][replace_indices]

filtered_params = read_csv(DIRS["cosmos"] / "filtered_params.csv")
check_columns_cosmos = ["e1", "e2", "mag", "hlr", "sersic"]
feat = filtered_params[check_columns_cosmos].values

catalog_selection = filtered_params[["indices", "sample"]].values

catalog_features_norm, test_points_norm = normalise(feat, target, SCALING)
diff = test_points_norm[:, numpy.newaxis, :] - catalog_features_norm[numpy.newaxis, :, :]

if NORMA == "euclid":
    matched_idx = euclidian_style(diff)
elif NORMA == "mahalanobis":
    matched_idx = maha_style(catalog_features_norm, diff)

matched_selection = catalog_selection[matched_idx]
matched_features = feat[matched_idx]

duplicates = check_dupicates(matched_selection)
path_save = DIRS["duplicates"] / f"{Path(str(version))}_duplicates_cosmos.npy"
numpy.save(file=path_save, arr=duplicates, allow_pickle=True)

lhs_gal[numpy.ix_(replace_indices, check_columns)] = matched_features
lhs_gal[replace_indices, 9:11] = matched_selection

In [ ]:
numpy.load(DIRS["duplicates"] / f"{Path(str(version))}_duplicates_cosmos.npy", allow_pickle=True)

In [ ]:
NB_OF_SAMPLES = 1
lhs_gal = draw_lhs_gal(False, 2222, NB_OF_SAMPLES * 10)
lhs_gal = project_lhs(hypercube=lhs_gal, ellips_style="gaussian", pixel_out=False)

PERCENTAGE_COSMOS = 50
nb_galaxies_cosmos = int(PERCENTAGE_COSMOS * NB_OF_SAMPLES * 10 / 100)
numpy.random.seed(22)
replace_indices = numpy.sort(
    numpy.random.choice(a=NB_OF_SAMPLES * 10, size=nb_galaxies_cosmos, replace=False)
)

In [ ]:
results_features = {}
results_selection = {}
for SCALING in ["min_max", "z_score"]:
    for NORMA in ["euclid", "mahalanobis"]:
        new_columns = numpy.zeros((lhs_gal.shape[0], 2), dtype=lhs_gal.dtype) - 1
        lhs_gal = numpy.hstack((lhs_gal, new_columns))

        check_columns = [0, 1, 4, 5, 6]  # e1 e2 mag hlr sersic
        target = lhs_gal[:, check_columns][replace_indices]

        filtered_params = read_csv(DIRS["cosmos"] / "filtered_params.csv")
        check_columns_cosmos = ["e1", "e2", "mag", "hlr", "sersic"]
        feat = filtered_params[check_columns_cosmos].values

        catalog_selection = filtered_params[["indices", "sample"]].values

        catalog_features_norm, test_points_norm = normalise(feat, target, SCALING)
        diff = test_points_norm[:, numpy.newaxis, :] - catalog_features_norm[numpy.newaxis, :, :]

        if NORMA == "euclid":
            matched_idx = euclidian_style(diff)
        elif NORMA == "mahalanobis":
            matched_idx = maha_style(catalog_features_norm, diff)

        matched_selection = catalog_selection[matched_idx]
        matched_features = feat[matched_idx]

        duplicates = check_dupicates(matched_selection)
        path_save = DIRS["duplicates"] / f"{Path(str(version))}_duplicates_cosmos.npy"
        numpy.save(file=path_save, arr=duplicates, allow_pickle=True)

        print(len(duplicates))

        # lhs_gal[numpy.ix_(replace_indices, check_columns)] = matched_features
        # lhs_gal[replace_indices, 9:11] = matched_selection

        results_features[(SCALING, NORMA)] = matched_features
        results_selection[(SCALING, NORMA)] = matched_selection


In [ ]:
def comparaison_3d(feat, target, results, with_all_catalog):
    target_text = [f"{i + 1}" for i in range(len(target))]

    colors = {
        ("min_max", "euclid"): "blue",
        ("min_max", "mahalanobis"): "green",
        ("z_score", "euclid"): "orange",
        ("z_score", "mahalanobis"): "purple",
    }

    fig = subplots.make_subplots(
        rows=1, cols=2, specs=[[{"type": "scene"}, {"type": "xy"}]], column_widths=[0.5, 0.5]
    )

    if with_all_catalog:
        fig.add_trace(
            graph_objects.Scatter3d(
                x=feat[:, 2],
                y=feat[:, 3],
                z=feat[:, 4],
                mode="markers",
                marker=dict(size=1, color="lightgray", opacity=0.3),
                name="feat",
            ),
            row=1,
            col=1,
        )

    fig.add_trace(
        graph_objects.Scatter3d(
            x=target[:, 2],
            y=target[:, 3],
            z=target[:, 4],
            mode="markers+text",
            marker=dict(size=3, color="red"),
            text=target_text,
            name="target",
        ),
        row=1,
        col=1,
    )

    for (SCALING, NORMA), matched in results.items():
        fig.add_trace(
            graph_objects.Scatter3d(
                x=matched[:, 2],
                y=matched[:, 3],
                z=matched[:, 4],
                mode="markers",
                marker=dict(size=4, color=colors[(SCALING, NORMA)]),
                name=f"{SCALING.upper()} + {NORMA.title()}",
            ),
            row=1,
            col=1,
        )

    if with_all_catalog:
        fig.add_trace(
            graph_objects.Scatter(
                x=feat[:, 0],
                y=feat[:, 1],
                mode="markers",
                marker=dict(size=3, color="lightgray", opacity=0.3),
                name="feat (2D)",
            ),
            row=1,
            col=2,
        )

    fig.add_trace(
        graph_objects.Scatter(
            x=target[:, 0],
            y=target[:, 1],
            mode="markers+text",
            text=target_text,
            marker=dict(color="red", size=6),
            name="Test points",
        ),
        row=1,
        col=2,
    )

    for (SCALING, NORMA), matched in results.items():
        fig.add_trace(
            graph_objects.Scatter(
                x=matched[:, 0],
                y=matched[:, 1],
                mode="markers",
                marker=dict(size=5, color=colors[(SCALING, NORMA)]),
                name=f"{SCALING.upper()} + {NORMA.title()} (2D)",
            ),
            row=1,
            col=2,
        )

    fig.update_xaxes(title_text="e1", row=1, col=2)
    fig.update_yaxes(title_text="e2", row=1, col=2)
    fig.update_layout(
        scene=dict(xaxis_title="mag", yaxis_title="hlr", zaxis_title="sersic"),
        width=1600,
        height=800,
        showlegend=True,
    )

    fig.show()


comparaison_3d(feat, target, results_features, with_all_catalog=True)

# matched

In [ ]:
from functions_cdfs import CDF_HLR_WRT_MAG, CDF_MAG, CDF_NS_WRT_MAG
import numpy
from functions_lhs import draw_lhs_gal

from scipy import interpolate

In [ ]:
def get_magnitude(array, inv_cdf_mag):
    inv_array = inv_cdf_mag(array)

    closest_half = numpy.around(inv_array * 2) / 2  # closest half categories
    closest_half[closest_half == 20] = 20.5  # to avoid 20.0

    return inv_array, closest_half

In [ ]:
# mag
magnitudes = CDF_MAG[:, 0]
cdf_values = CDF_MAG[:, 1]
inv_cdf_mag_fn = interpolate.interp1d(cdf_values, magnitudes, fill_value="extrapolate")

# ns
inv_cdfs_ns_fn = {}
for mag in magnitudes:
    case_data = CDF_NS_WRT_MAG[CDF_NS_WRT_MAG[:, 0] == mag]
    x_values = case_data[:, 1]
    cdf_values = case_data[:, 2]

    inv_cdfs_ns_fn[mag] = interpolate.interp1d(cdf_values, x_values, fill_value="extrapolate")

# hlr
inv_cdfs_hlr_fn = {}
for mag in magnitudes:
    case_data = CDF_HLR_WRT_MAG[CDF_HLR_WRT_MAG[:, 0] == mag]
    x_values = case_data[:, 1]
    cdf_values = case_data[:, 2]

    inv_cdfs_hlr_fn[mag] = interpolate.interp1d(cdf_values, x_values, fill_value="extrapolate")

In [ ]:
NB_OF_SAMPLES = 2000
hypercube = draw_lhs_gal(False, 22, NB_OF_SAMPLES * 10)
hypercube.shape

In [ ]:
hypercube[:, 4], closest_half = get_magnitude(hypercube[:, 4], inv_cdf_mag_fn)
hypercube[:, 4]


In [ ]:
def get_half_light_radius(lhs_gal, closest_half, inv_cdfs_hlr):
    for i, cas in enumerate(closest_half):
        value_lhs = lhs_gal[i]
        lhs_gal[i] = inv_cdfs_hlr[float(cas)](value_lhs)

    min_value = abs(lhs_gal.min())
    lhs_gal = lhs_gal + 2 * min_value + 0.05

    return lhs_gal


def get_sersic_index(lhs_gal, closest_half, inv_cdfs_ns):
    for i, cas in enumerate(closest_half):
        value_lhs = lhs_gal[i]
        lhs_gal[i] = inv_cdfs_ns[float(cas)](value_lhs)

    ns_possible_values = numpy.unique(CDF_NS_WRT_MAG[:, 1])

    lhs_gal = ns_possible_values[numpy.abs(lhs_gal[:, None] - ns_possible_values).argmin(axis=1)]
    lhs_gal = numpy.clip(lhs_gal, 0.3, 6.2)

    return lhs_gal